In [1]:
import pandas as pd
import numpy as np
import re
import html
import unicodedata
import json
import warnings
from openai import OpenAI

warnings.filterwarnings('ignore')

In [2]:
import os
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

# Inicializar cliente de OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4o-mini"
print(f'Cliente OpenAI inicializado. Modelo: {MODEL}')

Cliente OpenAI inicializado. Modelo: gpt-4o-mini


## Carga dataset

In [3]:
with open('../data/processed/supermercado_nutricion.json', 'r', encoding='utf-8') as file :
  df_raw = pd.read_json(file)
print (len(df_raw))
print (df_raw.shape[1])
df_raw.head(3)

11305
18


,nombre,departamento,categoria,subcategoria,marca,contenido,precio,descripcion,unidad_medicion,kcal,proteina_g,carbohidratos_g,azucares_g,grasas_totales_g,grasas_saturadas_g,fibra_g,sodio_mg,notas_nutricionales
0,Vino Panul tinto cabernet sauvignon clásico bo...,Supermercado,Vinos Y Licores,Vino,PANUL,750ml,21990.0,Vino Panul tinto cabernet sauvignon clásico bo...,por 100 ml,85.0,0.1,2.6,0.6,0.0,0.0,0.0,5.0,Tinto/blanco ~85 kcal; ~12% alcohol; contiene ...
1,Vino Tinto Merlot Clásico Panul botella x 750 ml,Supermercado,Vinos Y Licores,Vino,PANUL,750ml,21990.0,Nuestros Vinos | Ficha técnica y notas de cata...,por 100 ml,85.0,0.1,2.6,0.6,0.0,0.0,0.0,5.0,Tinto/blanco ~85 kcal; ~12% alcohol; contiene ...
2,Vino tinto Peñasol botella x750ml,Supermercado,Vinos Y Licores,Vino,PENASOL,750ml,16990.0,Vino Tinto Peñasol x 750ml. El exceso de alcoh...,por 100 ml,85.0,0.1,2.6,0.6,0.0,0.0,0.0,5.0,Tinto/blanco ~85 kcal; ~12% alcohol; contiene ...


## EDA

In [4]:
# Tipos de datos
print(df_raw.dtypes.to_string())

nombre                     str
departamento               str
categoria                  str
subcategoria               str
marca                      str
contenido                  str
precio                 float64
descripcion                str
unidad_medicion            str
kcal                   float64
proteina_g             float64
carbohidratos_g        float64
azucares_g             float64
grasas_totales_g       float64
grasas_saturadas_g     float64
fibra_g                float64
sodio_mg               float64
notas_nutricionales        str


In [5]:
# Valores nulos
nulos = df_raw.isnull().sum()
pct   = (nulos / len(df_raw) * 100).round(2)
nulos_df = pd.DataFrame({'nulos': nulos, 'pct_%': pct})
nulos_df = nulos_df[nulos_df['nulos'] > 0].sort_values('nulos', ascending=False)

print('COLUMNAS CON VALORES NULOS')
print(nulos_df.to_string())

COLUMNAS CON VALORES NULOS
                    nulos  pct_%
precio                226   2.00
kcal                   36   0.32
grasas_totales_g       36   0.32
proteina_g             36   0.32
carbohidratos_g        36   0.32
azucares_g             36   0.32
fibra_g                36   0.32
grasas_saturadas_g     36   0.32
sodio_mg               36   0.32
descripcion            20   0.18


In [6]:
# Distribución de categorías
print('CATEGORIAS')
print(df_raw['categoria'].value_counts().to_string())

print('\nTOP 20 SUBCATEGORIAS')
print(df_raw['subcategoria'].value_counts().head(20).to_string())

CATEGORIAS
categoria
Despensa                          2606
Lácteos, Huevos Y Refrigerados    1652
Vinos Y Licores                   1256
Pasabocas                         1059
Frutas Y Verduras                  993
Dulces Y Postres                   690
Panadería Y Pastelería             678
Bebidas                            625
Platos Preparados                  512
Carne Y Pollo                      506
Charcutería                        449
Pescados Y Mariscos                279

TOP 20 SUBCATEGORIAS
subcategoria
Hortalizas                           606
Vino                                 510
Salsas Y Vinagres                    430
Yogurt                               385
Papas Fritas Y Paquetes              372
Panadería Empacada                   366
Queso                                344
Enlatados Y Conservas                304
Condimentos, Caldos Y Sal            275
Café                                 263
Confitería                           245
Chocolatería             

In [7]:
# Estadísticas de precio
print('ESTADISTICAS DE PRECIO (COP)')

print(df_raw['precio'].describe().apply(lambda x: f'{x:,.0f}').to_string())

precio_cero = (df_raw['precio'] == 0).sum()
precio_alto = (df_raw['precio'] > 500_000).sum()
print(f'\nProductos con precio 0 COP:       {precio_cero}')
print(f'Productos con precio > 500.000 COP: {precio_alto}')

ESTADISTICAS DE PRECIO (COP)
count       11,079
mean        30,272
std         65,525
min            290
25%          7,841
50%         15,590
75%         30,890
max      2,345,990

Productos con precio 0 COP:       0
Productos con precio > 500.000 COP: 25


In [8]:
# Estadísticas nutricionales
cols_nutri = ['kcal', 'proteina_g', 'carbohidratos_g', 'azucares_g',
              'grasas_totales_g', 'grasas_saturadas_g', 'fibra_g', 'sodio_mg']

print('ESTADISTICAS NUTRICIONALES:')
print(df_raw[cols_nutri].describe().round(2).to_string())

ESTADISTICAS NUTRICIONALES:
           kcal  proteina_g  carbohidratos_g  azucares_g  grasas_totales_g  grasas_saturadas_g  fibra_g  sodio_mg
count  11269.00    11269.00         11269.00    11269.00          11269.00            11269.00  11269.0  11269.00
mean     241.65        7.50            26.83       10.06             10.09                3.68      1.9    323.55
std      178.66        7.91            29.30       18.99             16.58                5.99      3.9    604.91
min        0.00        0.00             0.00        0.00              0.00                0.00      0.0      0.00
25%       85.00        1.00             2.60        0.50              0.30                0.00      0.0     10.00
50%      231.00        5.00            10.80        3.00              3.00                0.80      0.5     55.00
75%      370.00       12.00            51.00       10.00             14.00                6.00      2.5    480.00
max      884.00       31.00            99.50       99.50    

In [9]:
# Duplicados
dup_filas   = df_raw.duplicated().sum()
dup_nombres = df_raw['nombre'].duplicated().sum()

print(f'Filas completamente duplicadas: {dup_filas}')
print(f'Nombres de producto duplicados: {dup_nombres}')

if dup_nombres > 0:
    display(
        df_raw[df_raw['nombre'].duplicated(keep=False)]
        .sort_values('nombre')[['nombre', 'subcategoria', 'marca', 'precio']]
        .head(8)
    )

Filas completamente duplicadas: 0
Nombres de producto duplicados: 23


,nombre,subcategoria,marca,precio
6764,Aceite Farchioni oliva extra virgen albahaca x...,Aceite,FARCHIONI,12990.0
6754,Aceite Farchioni oliva extra virgen albahaca x...,Aceite,FARCHIONI,40690.0
6738,Aceite Farchioni oliva extra virgen limón x250ml,Aceite,FARCHIONI,38890.0
6765,Aceite Farchioni oliva extra virgen limón x250ml,Aceite,FARCHIONI,14990.0
6752,Aceite Farchioni oliva extra virgen x500ml,Aceite,FARCHIONI,57790.0
6733,Aceite Farchioni oliva extra virgen x500ml,Aceite,FARCHIONI,52720.0
6734,Aceite Sublime oliva Español x500ml,Aceite,SUBLIME,39400.0
6696,Aceite Sublime oliva Español x500ml,Aceite,SUBLIME,37000.0


## Limpieza de datos

In [10]:
def limpiar_texto(valor):

    if pd.isna(valor):
        return valor
    s = str(valor)
    s = html.unescape(s)
    s = s.replace('\x82', "'").replace('\x98', "'")
    s = unicodedata.normalize('NFC', s)
    s = s.replace('\u2018', "'").replace('\u2019', "'")
    s = s.replace('\u201c', '"').replace('\u201d', '"')
    s = s.replace('\u00b4', "'").replace('\xa0', ' ')
    s = re.sub(r'  +', ' ', s).strip()
    return s


def limpiar_marca(valor):
    # Limpia y estandariza nombres de marcas a MAYÚSCULAS
    if pd.isna(valor):
        return valor
    s = limpiar_texto(valor)
    correcciones = {
        'FLOR DE CA\u00f1A': 'FLOR DE CA\u00d1A',
        'COMPLET\u00edSIMO':  'COMPLET\u00cdSIMO',
        'MAMA-\u00ecA':       'MAMA-\u00cdA',
    }
    no_es_marca = {'Patacon toston paquete por 5 u'}
    if s in no_es_marca:
        return 'SIN MARCA'
    s = correcciones.get(s, s)
    return s.upper()


def normalizar_contenido(valor):
    #Normaliza el campo 'contenido': '750 ml' -> '750ml', '500gr' -> '500g'."""
    if pd.isna(valor):
        return valor
    s = str(valor).strip()
    s = re.sub(r'([\d.,]+)\s+([a-zA-Z]+)', r'\1\2', s)
    s = re.sub(r'([\d.,]+)([a-zA-Z]+)', lambda m: m.group(1) + m.group(2).lower(), s)
    s = re.sub(r'(\d)grs?\b', r'\1g', s)
    s = re.sub(r'(\d)lts?\b', r'\1l', s)
    return s.strip()

In [11]:
# APLICAR LIMPIEZA
df = df_raw.copy()

# Texto libre
for col in ['nombre', 'descripcion', 'notas_nutricionales']:
    df[col] = df[col].apply(limpiar_texto)

# Marca
df['marca'] = df['marca'].apply(limpiar_marca)

# Contenido
df['contenido'] = df['contenido'].apply(normalizar_contenido)

# Precio: 0 como NaN
df['precio'] = pd.to_numeric(df['precio'], errors='coerce')
df.loc[df['precio'] == 0, 'precio'] = np.nan

# Columnas nutricionales: asegurar tipo numérico
cols_nutri = ['kcal', 'proteina_g', 'carbohidratos_g', 'azucares_g',
              'grasas_totales_g', 'grasas_saturadas_g', 'fibra_g', 'sodio_mg']
for col in cols_nutri:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Eliminar duplicados exactos
n_antes = len(df)
df = df.drop_duplicates()

print(f'Filas eliminadas por duplicado exacto: {n_antes - len(df)}')
print(f'Dataset limpio: {len(df):,} filas')

Filas eliminadas por duplicado exacto: 0
Dataset limpio: 11,305 filas


## Preparación del DataFrame para el agente

In [12]:
# Renombrar columnas clave
df.rename(columns={'precio': 'precio_cop', 'kcal': 'calorias_kcal'}, inplace=True)

# Flags dietéticos basados en subcategoría
SUBCATS_VEGANO = {
    'Frutas', 'Hortalizas', 'Papas, Yucas Y Patacones', 'Fruta Y Ensaladas',
    'Verduras Congeladas', 'Hierbas', 'Pulpa De Fruta', 'Arroz Y Granos',
    'Aceite', 'Enlatados Y Conservas', 'Encurtidos Y Otros',
    'Azúcar, Endulzantes Y Panelas', 'Té, Infusiones E Instantáneos'
}

SUBCATS_VEGETARIANO = SUBCATS_VEGANO | {
    'Yogurt', 'Queso', 'Quesos Especializados', 'Leche', 'Leches Vegetales',
    'Huevos', 'Mantequilla', 'Margarina', 'Arequipe',
    'Crema Leche Y Leche Condensada', 'Postres Refrigerados', 'Kumis', 'Avena',
    'Galletas Dulces', 'Galletas Saladas', 'Galletas Dietéticas Y Saludables',
    'Panadería Empacada', 'Panadería Artesanal', 'Helados', 'Confitería',
    'Chocolatería', 'Cereales', 'Pastas', 'Arepas', 'Chocolate De Mesa',
    'Bebida Achocolatada En Polvo', 'Mermelada Y Cremas De Untar',
    'Frutos Secos', 'Dulces Típicos', 'Repostería', 'Pastelería'
}

df['apto_vegetariano'] = df['subcategoria'].isin(SUBCATS_VEGETARIANO)
df['apto_vegano']      = df['subcategoria'].isin(SUBCATS_VEGANO)

# Solo productos con precio
df_agente = df.dropna(subset=['precio_cop']).copy()

print(f'DataFrame para el agente: {len(df_agente):,} productos con precio')
print(f'Vegetarianos: {df_agente["apto_vegetariano"].sum():,}')
print(f'Veganos:      {df_agente["apto_vegano"].sum():,}')
df_agente[['nombre','subcategoria','precio_cop','calorias_kcal','apto_vegetariano','apto_vegano']].head(5)

DataFrame para el agente: 11,079 productos con precio
Vegetarianos: 6,138
Veganos:      2,014


,nombre,subcategoria,precio_cop,calorias_kcal,apto_vegetariano,apto_vegano
0,Vino Panul tinto cabernet sauvignon clásico bo...,Vino,21990.0,85.0,False,False
1,Vino Tinto Merlot Clásico Panul botella x 750 ml,Vino,21990.0,85.0,False,False
2,Vino tinto Peñasol botella x750ml,Vino,16990.0,85.0,False,False
3,Vino blanco Panul Sauvignon Blanc x750ml,Vino,21990.0,85.0,False,False
4,Vino tinto Carmenere Clásico Panul botella x 7...,Vino,21990.0,85.0,False,False


In [13]:
def parse_user_intent(user_request: str) -> dict:
    """
    GPT-4o mini para extraer presupuesto y preferencias dietéticas
    a partir del texto libre del usuario.
    """
    system_prompt = """
Eres un extractor de información de solicitudes de compras en supermercado.
A partir del texto del usuario, extrae ÚNICAMENTE los parámetros que se mencionen explícita o implícitamente.

Devuelve SIEMPRE un JSON válido con exactamente estas claves (usa null si no se menciona):
{
  "budget": <número en COP o null>,
  "apto_vegetariano": <true/false/null>,
  "apto_vegano": <true/false/null>,
  "azucares_g_max": <número o null>,
  "grasas_totales_g_max": <número o null>,
  "proteina_g_min": <número o null>,
  "calorias_kcal_max": <número o null>,
  "sodio_mg_max": <número o null>,
  "num_productos": <entero entre 3 y 15, por defecto 8>,
  "contexto": <resumen breve de la solicitud en español, máx 60 palabras>
}

Reglas:
- Si dice "vegano", poner apto_vegano: true Y apto_vegetariano: true
- Si dice "vegetariano", poner apto_vegetariano: true
- Si dice "bajo en azúcar" sin cifra, usar azucares_g_max: 5
- Si dice "bajo en grasa" sin cifra, usar grasas_totales_g_max: 10
- Si dice "alto en proteína" sin cifra, usar proteina_g_min: 15
- No incluyas texto fuera del JSON.
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_request}
        ],
        temperature=0,       #determinístico para parseo
        max_tokens=300,
        response_format={"type": "json_object"}  # garantiza JSON válido
    )

    return json.loads(response.choices[0].message.content)


print('parse_user_intent() definida OK')

parse_user_intent() definida OK


In [14]:
def filter_catalog(intent: dict, max_products: int = 20) -> pd.DataFrame:
    """
    Filtra df_agente según los criterios extraídos por parse_user_intent().

    Args:
        intent (dict): Resultado de parse_user_intent()
        max_products (int): Máximo de productos a devolver al generador
                            (muestra diversa por subcategoría)

    Returns:
        pd.DataFrame: Subconjunto filtrado del catálogo
    """
    filt = df_agente.copy()

    # Filtro de presupuesto
    if intent.get('budget'):
        filt = filt[filt['precio_cop'] <= intent['budget']]

    # Filtros dietéticos
    if intent.get('apto_vegano'):
        filt = filt[filt['apto_vegano'] == True]
    elif intent.get('apto_vegetariano'):
        filt = filt[filt['apto_vegetariano'] == True]

    # Filtros nutricionales con tolerancia a NaN
    nutri_filters = {
        'azucares_g_max':        ('azucares_g',        'max'),
        'grasas_totales_g_max':  ('grasas_totales_g',  'max'),
        'proteina_g_min':        ('proteina_g',         'min'),
        'calorias_kcal_max':     ('calorias_kcal',      'max'),
        'sodio_mg_max':          ('sodio_mg',           'max'),
    }

    for intent_key, (col, tipo) in nutri_filters.items():
        val = intent.get(intent_key)
        if val is not None and col in filt.columns:
            if tipo == 'max':
                filt = filt[filt[col].isna() | (filt[col] <= val)]
            else:  # min
                filt = filt[filt[col].isna() | (filt[col] >= val)]

    if filt.empty:
        return filt

    # Si hay presupuesto, ordenar por precio para favorecer productos económicos
    if intent.get('budget'):
      filt = filt.sort_values('precio_cop')

    # Muestreo diverso por subcategoría (evita que el contexto sea mono-categoría)
    n_cats = filt['subcategoria'].nunique()
    per_cat = max(1, max_products // max(n_cats, 1))

    sample = (
        filt.groupby('subcategoria', group_keys=False)
            .apply(lambda g: g.sample(min(len(g), per_cat), random_state=42))
    )

    # Si el muestreo dio menos de max_products, completar aleatoriamente
    if len(sample) < max_products and len(filt) > len(sample):
        extra = filt.drop(sample.index).sample(
            min(max_products - len(sample), len(filt) - len(sample)),
            random_state=42
        )
        sample = pd.concat([sample, extra])

    return sample.head(max_products)


print('filter_catalog() definida OK')

filter_catalog() definida OK


In [15]:
def generate_basket(user_request: str, intent: dict, products_df: pd.DataFrame) -> str:
    """
    GPT-4o mini para generar una canasta de compras personalizada
    con precio total estimado y justificación nutricional por producto.

    Args:
        user_request (str): Solicitud original del usuario
        intent (dict): Intención extraída por parse_user_intent()
        products_df (pd.DataFrame): Productos filtrados por filter_catalog()

    Returns:
        str: Respuesta del agente con la canasta sugerida
    """
    # Preparar resumen de restricciones activas
    restricciones = []
    if intent.get('budget'):
        restricciones.append(f"Presupuesto máximo: ${intent['budget']:,.0f} COP")
    if intent.get('apto_vegano'):
        restricciones.append("Dieta: VEGANA")
    elif intent.get('apto_vegetariano'):
        restricciones.append("Dieta: VEGETARIANA")
    if intent.get('azucares_g_max'):
        restricciones.append(f"Azúcares ≤ {intent['azucares_g_max']} g/100g")
    if intent.get('grasas_totales_g_max'):
        restricciones.append(f"Grasas totales ≤ {intent['grasas_totales_g_max']} g/100g")
    if intent.get('proteina_g_min'):
        restricciones.append(f"Proteína ≥ {intent['proteina_g_min']} g/100g")
    if intent.get('calorias_kcal_max'):
        restricciones.append(f"Calorías ≤ {intent['calorias_kcal_max']} kcal/100g")
    if intent.get('sodio_mg_max'):
        restricciones.append(f"Sodio ≤ {intent['sodio_mg_max']} mg/100g")

    restricciones_str = '\n'.join(f'  • {r}' for r in restricciones) if restricciones else '  • Sin restricciones especiales'
    num_productos = intent.get('num_productos', 8)

    # Preparar tabla de productos disponibles
    cols_display = [
        'nombre', 'marca', 'subcategoria', 'precio_cop',
        'calorias_kcal', 'proteina_g', 'azucares_g',
        'grasas_totales_g', 'fibra_g', 'contenido'
    ]
    available_cols = [c for c in cols_display if c in products_df.columns]
    products_str = products_df[available_cols].to_markdown(index=False)

    system_prompt = f"""
Eres un asistente de compras inteligente.
Tu objetivo es ayudar a los usuarios a armar canastas de productos saludables,
económicas y adaptadas a sus necesidades.

CONTEXTO DE LA SOLICITUD:
{intent.get('contexto', user_request)}

RESTRICCIONES ACTIVAS:
{restricciones_str}

INSTRUCCIONES — sigue estas reglas estrictamente:
1. Selecciona exactamente {num_productos} productos de la lista proporcionada.
2. Elige productos variados de diferentes subcategorías cuando sea posible.
3. IMPORTANTE: la suma de precios de los productos seleccionados NO debe superar
   el presupuesto indicado. Si no es posible seleccionar {num_productos} productos
   dentro del presupuesto, selecciona los que quepan y explica brevemente por qué.
4. Para cada producto seleccionado indica en una sola línea:
   - Nombre, precio en COP y justificación nutricional breve
5. Al final muestra ÚNICAMENTE:
   - PRECIO TOTAL: suma exacta de los productos seleccionados
   - RESUMEN NUTRICIONAL: calorías promedio, proteína promedio, azúcares promedio
6. No hagas iteraciones ni ajustes dentro de tu respuesta. Decide de una vez y muestra nuevamente la data final con el ajuste.
  - Si hay ajuste para cada producto seleccionado indica en una sola línea: Nombre, precio en COP y justificación nutricional breve
7. Los valores nutricionales son por 100g (sólidos) o por 100ml (líquidos).
8. Responde siempre en español.
"""

    user_prompt = f"""
Solicitud del usuario: "{user_request}"

Productos disponibles en el catálogo:
{products_str}

Por favor, arma la canasta de compras.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=1500
    )

    return response.choices[0].message.content


print('generate_basket() definida OK')

generate_basket() definida OK


In [ ]:
def shopping_agent(user_request: str, verbose: bool = True) -> str:
    """
    Función principal del agente de compras.
    Orquesta las tres etapas: parseo, filtrado, generación.

    Args:
        user_request (str): Solicitud en lenguaje natural del usuario
        verbose (bool): Si True, muestra la intención extraída y stats del filtrado

    Returns:
        str: Canasta de compras generada por el agente
    """

    print(f'AGENTE DE COMPRAS')
    print(f'Usuario: "{user_request}"')

    # Etapa 1: Parseo de intención
    print('\n  Etapa 1/3: Interpretando solicitud')
    intent = parse_user_intent(user_request)

    if verbose:
        print('   Intención extraída:')
        for k, v in intent.items():
            if v is not None and k != 'contexto':
                print(f'     {k}: {v}')
        if intent.get('contexto'):
            print(f'   Contexto: {intent["contexto"]}')

    # Etapa 2: Filtrado del catálogo
    print('\n Etapa 2/3: Filtrando catálogo')
    filtered = filter_catalog(intent, max_products=20)

    if filtered.empty:
        msg = (
            'Lo siento, no encontré productos que cumplan todos los criterios. '
            'Te sugiero relajar alguna restricción (presupuesto, dieta o nutrición).'
        )
        print(f'     {msg}')
        return msg

    if verbose:
        print(f'   Productos encontrados: {len(filtered):,}')        
        precio_rango = f"${filtered['precio_cop'].min():,.0f} – ${filtered['precio_cop'].max():,.0f} COP"
        print(f'   Rango de precio: {precio_rango}')

    # Etapa 3: Generación de canasta
    print('\n Etapa 3/3: Generando canasta con GPT-4o mini')
    respuesta = generate_basket(user_request, intent, filtered)

    print('CANASTA SUGERIDA:')
    print(respuesta)

    return respuesta

print('shopping_agent() definida OK')
print('\n El agente está listo. Usa: shopping_agent("tu solicitud aquí")')

shopping_agent() definida OK

 El agente está listo. Usa: shopping_agent("tu solicitud aquí")


## Ejemplos de uso


In [21]:
#  Ejemplo 1: Canasta vegetariana con presupuesto

resp_1 = shopping_agent(
    "Necesito una canasta vegetariana para la semana con un presupuesto de 50.000 pesos"
)

AGENTE DE COMPRAS
Usuario: "Necesito una canasta vegetariana para la semana con un presupuesto de 50.000 pesos"

  Etapa 1/3: Interpretando solicitud
   Intención extraída:
     budget: 50000
     apto_vegetariano: True
     num_productos: 8
   Contexto: Solicitud de canasta vegetariana para la semana con un presupuesto específico.

 Etapa 2/3: Filtrando catálogo
   Productos encontrados: 20
   Rango de precio: $3,490 – $43,990 COP

 Etapa 3/3: Generando canasta con GPT-4o mini
CANASTA SUGERIDA:
Aquí tienes una canasta vegetariana para la semana, ajustada al presupuesto de $50,000 COP:

1. **Arepas de yuca Cajoneritas La Cajonera con queso x12und x300g** - **15,690 COP**: Buena fuente de carbohidratos y proteína vegetal.
2. **Avena Colanta natural x1100ml** - **6,850 COP**: Rica en fibra y baja en grasas, ideal para el desayuno.
3. **Cereal Quaker Oats Squares x411g** - **16,990 COP**: Fuente de energía con buena cantidad de proteína y fibra.
4. **Arveja La Coruña Natural x 600g** - **

In [22]:
#  Ejemplo 2: Desayuno saludable para dos personas
resp_2 = shopping_agent(
    "Busco productos para hacer un desayuno saludable para dos personas, "
    "alto en proteína y bajo en azúcar, sin superar 40.000 pesos"
)

AGENTE DE COMPRAS
Usuario: "Busco productos para hacer un desayuno saludable para dos personas, alto en proteína y bajo en azúcar, sin superar 40.000 pesos"

  Etapa 1/3: Interpretando solicitud
   Intención extraída:
     budget: 40000
     azucares_g_max: 5
     proteina_g_min: 15
     num_productos: 8
   Contexto: Solicitud de productos para un desayuno saludable, alto en proteína y bajo en azúcar para dos personas.

 Etapa 2/3: Filtrando catálogo
   Productos encontrados: 20
   Rango de precio: $4,800 – $39,500 COP

 Etapa 3/3: Generando canasta con GPT-4o mini
CANASTA SUGERIDA:
Aquí tienes la canasta de productos seleccionados para un desayuno saludable, alto en proteína y bajo en azúcar para dos personas:

1. **Nuggets Zenu pollo apanado x10und x160g** - 10,720 COP: Alto en proteína (17 g) y bajo en azúcares (0.5 g), ideal como fuente de proteína.
2. **Avena Pro Alqueria Autentica con extra Magnesio x4unds** - 13,490 COP: Contiene 16.9 g de proteína y solo 1 g de azúcares, excele

In [23]:
#  Ejemplo 3: Canasta vegana
resp_3 = shopping_agent(
    "Soy vegano y quiero una canasta variada para toda la semana. "
    "No tengo restricción de presupuesto pero prefiero opciones económicas."
)

AGENTE DE COMPRAS
Usuario: "Soy vegano y quiero una canasta variada para toda la semana. No tengo restricción de presupuesto pero prefiero opciones económicas."

  Etapa 1/3: Interpretando solicitud
   Intención extraída:
     apto_vegetariano: True
     apto_vegano: True
     num_productos: 8
   Contexto: Solicitud de una canasta variada para toda la semana, sin restricción de presupuesto pero con preferencia por opciones económicas.

 Etapa 2/3: Filtrando catálogo
   Productos encontrados: 20
   Rango de precio: $4,890 – $50,890 COP

 Etapa 3/3: Generando canasta con GPT-4o mini
CANASTA SUGERIDA:
Aquí tienes una canasta variada para toda la semana, con productos saludables y económicos, adaptada a una dieta vegana:

1. **Cous Cous Divella medio x500g** - **28,090 COP**: Rico en carbohidratos complejos y fibra, ideal para proporcionar energía sostenida.
2. **Ensalada Arcoíris Hortifresco 150g** - **10,790 COP**: Mezcla de verduras que aporta vitaminas, minerales y fibra, contribuyendo

In [ ]:
# Ejemplo 4: Restricciones muy estrictas
resp_4 = shopping_agent(
    "Necesito productos veganos con menos de 0.5 gramos de grasa y precio máximo 200 pesos"
)

AGENTE DE COMPRAS
Usuario: "Necesito productos veganos con menos de 0.5 gramos de grasa y precio máximo 200 pesos"

  Etapa 1/3: Interpretando solicitud


In [ ]:
# Ejemplo 5: Solicitud en lenguaje muy natural
resp_5 = shopping_agent(
    "Estoy a dieta, quiero comer sano pero rico. Tengo 40 mil pesos. "
    "Prefiero cosas con poca grasa y que sean llenadores."
)